# 1부두 전처리

## 1. 파일 읽기 → 원본 확인

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
DATA = Path('../data')
# OUT = Path('output_preprocess')
# OUT.mkdir(exist_ok=True)

def read_csv(path):
    for encoding in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str).fillna('')
        except UnicodeDecodeError:
            continue
    raise ValueError('파일 인코딩을 확인하세요.')

# record = read_csv(DATA / 'record_except_168.csv') # 전처리 전 데이터 파일
record = read_csv(DATA / 'record.csv')
port1 = read_csv(DATA / 'port1.csv')

print('record:', record.shape, 'port1:', port1.shape)
display(record.head())
display(port1.head())
# assert len(record)==19981 and len(port1)==5336, '수정 전 원본을 넣어주세요.'

record: (19989, 13) port1: (2928, 12)


,호출부호,선명,입항횟수,총톤수,입항일시,출항일시,CIQ수속일자,수리일시,MRN 번호,계선장소,차항지,전출항지,재항시간
0,3FRF8,SONG YUN HE,2022.0,"16,737",2022-01-01 00:48:00,2022-01-03 01:52:00,2022-01-03 1:52,2022-01-03 15:58,22COHES152E,신항 5부두 3선석,QINGDAO,광양,49.06666666666667
1,VRNA9,SEASPAN THAMES,2022.0,"112,967",2022-01-01 02:00:00,2022-01-02 17:43:00,2022-01-02 17:43,2022-01-03 11:18,22HLCU4202E,신항 3부두 4선석,VANCOUVER,SHANGHAI,39.71666666666667
2,VRRP8,CMA CGM MUMBAI,2022.0,"112,967",2022-01-01 03:00:00,2022-01-02 04:06:00,2022-01-02 4:00,2022-01-03 11:22,22CMAL0030E,신항 5부두 2선석,MANZANILLO,SHANGHAI,25.1
3,D5TL4,MSC JEWEL,2022.0,"140,976",2022-01-01 03:22:00,2022-01-03 05:00:00,2022-01-03 5:00,2022-01-03 11:19,22MSCU0004E,신항 1부두 3선석,MANZANILLO,HONG KONG,49.63333333333333
4,A8YT7,MALIAKOS,2022.0,"41,391",2022-01-01 03:35:00,2022-01-01 16:45:00,2022-01-01 16:00,2022-01-03 11:07,22SMLME016E,신항 2부두 1선석,VANCOUVER,SHANGHAI,13.166666666666666


,선석,선사,모선항차,선사항차,선명,ROUTE,반입마감시한,접안(예정)일시,출항(예정)일시,양하,적하,Shift
0,T3(P),MSC,MJEW008,FA147A/FA147A,MSC JEWEL,ASPA2E,2021-12-31 16:00,2022-01-01 3:40,2022-01-03 5:00,1263,2905,2
1,T1(P),COS,COSP001,090E/090E,COSCO SAO PAULO,WSA3,2021-12-31 20:00,2022-01-01 7:20,2022-01-02 7:00,402,1066,0
2,T2(P),HLC,COPX001,2140W/2152E,COPIAPO,AN1,2022-01-01 22:00,2022-01-02 8:10,2022-01-04 12:00,1471,1998,108
3,T1(P),MSC,MELN003,UK150A/UK150A,MSC ELAINE,CHINK,2022-01-01 22:00,2022-01-02 9:10,2022-01-03 20:00,838,1281,2
4,T3(P),MSC,MDES002,FI144R/FI150A,MSC DESIREE,AS2,2022-01-02 19:00,2022-01-03 7:20,2022-01-04 20:00,1139,1900,570


## 2. 1부두와 분석 기간 추출 → 건수 확인

In [2]:
# r : record 에서 22년~25년 사이 1부두 데이터
r = record[record['계선장소'].str.contains('신항 1부두')].copy()
r['rid'] = r.index
r['입항'] = pd.to_datetime(r['입항일시'])
r['출항'] = pd.to_datetime(r['출항일시'])
r = r[r['입항'].dt.year.between(2022,2025)].copy()

# p : port1 에서 22년~25년 사이 1부두 데이터
p = port1.copy()
p['pid'] = p.index
p['접안'] = pd.to_datetime(p['접안(예정)일시'])
p['터미널출항'] = pd.to_datetime(p['출항(예정)일시'])
p = p[p['접안'].dt.year.between(2022,2025)].copy()

print('record 1부두:', len(r), 'port1 기간 내:', len(p))
display(r[['선명','입항','출항','계선장소']].head(10))

record 1부두: 2928 port1 기간 내: 2928


,선명,입항,출항,계선장소
3,MSC JEWEL,2022-01-01 03:22:00,2022-01-03 05:00:00,신항 1부두 3선석
7,COSCO SAO PAULO,2022-01-01 07:10:00,2022-01-02 06:55:00,신항 1부두 1선석
17,COPIAPO,2022-01-02 08:15:00,2022-01-04 12:05:00,신항 1부두 2선석
18,MSC ELAINE,2022-01-02 08:58:00,2022-01-03 19:55:00,신항 1부두 1선석
30,MSC DESIREE,2022-01-03 07:10:00,2022-01-04 20:20:00,신항 1부두 3선석
39,CROATIA,2022-01-03 21:57:00,2022-01-05 05:05:00,신항 1부두 1선석
48,XIN BIN ZHOU,2022-01-04 14:00:00,2022-01-05 12:00:00,신항 1부두 2선석
57,MSC NORA,2022-01-05 08:43:00,2022-01-06 02:15:00,신항 1부두 1선석
58,MSC ELISA,2022-01-05 09:05:00,2022-01-07 09:32:00,신항 1부두 3선석
66,CONTI ANNAPURNA,2022-01-06 04:15:00,2022-01-08 07:25:00,신항 1부두 1선석



1. 선박이름
2. 입항or출항이 같거나 비슷(6시간이내or3시간 이내)
3. 입항~출항 사이에 접안or출항 있는거
4. record 기준으로

이렇게 보고 얘네랑 다른 애들 비교했는데

선박명 거의 비슷한 애들 있었어서 갸들 record에 맞췄나? 무튼 그렇게 해서 넣었(던거같)고

그다음에 둘다 안겹치는 애들 (record에만 있거나 port1에만 있는거) 뺐고

저거 record는 1개인데 port는 2개인거 분리하고 저거 집어넣음


## 3. 선박명 정리 → 표기 차이 확인
대문자·공백·기호를 정리합니다. 아래 9개 이름 대응은 앞서 검토한 표기/선명 변경 사항입니다. 유사도만으로 동일 선박이라고 자동 확정하지 않습니다. 이 작은 이름 사전만 검토 결과를 사용하며, 항차와 날짜는 모두 파일에서 계산합니다.

In [3]:
r['이름키'] = r['선명'].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
p['이름키'] = p['선명'].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
aliases={'MSCKYMEA':'MSCKYMEAII','MSCLISBON':'MSCLISBONIX','MSCTAMPICO':'MSCTAMPICOV','ZIMDANUBE':'DANUBE','ZIMSPARROW':'SPARROW','ZIMGANGES':'GANGES','MSCRANIAVIII':'MSCRANIA','TAYMA':'TAYMAEXPRESS','MONTEVIDEOEXPRESS':'NAVEGANTESEXPRESS'}
r['대응키'] = r['이름키'].replace(aliases)
p['대응키'] = p['이름키'].replace(aliases)
display(pd.DataFrame(aliases.items(), columns=['기존 이름키','통일 이름키']))
display(r.loc[r['이름키'] != r['대응키'], ['선명','이름키','대응키']].drop_duplicates())

,기존 이름키,통일 이름키
0,MSCKYMEA,MSCKYMEAII
1,MSCLISBON,MSCLISBONIX
2,MSCTAMPICO,MSCTAMPICOV
3,ZIMDANUBE,DANUBE
4,ZIMSPARROW,SPARROW
5,ZIMGANGES,GANGES
6,MSCRANIAVIII,MSCRANIA
7,TAYMA,TAYMAEXPRESS
8,MONTEVIDEOEXPRESS,NAVEGANTESEXPRESS


,선명,이름키,대응키
205,MSC KYMEA,MSCKYMEA,MSCKYMEAII
503,MSC LISBON,MSCLISBON,MSCLISBONIX
662,MONTEVIDEO EXPRESS,MONTEVIDEOEXPRESS,NAVEGANTESEXPRESS
734,TAYMA,TAYMA,TAYMAEXPRESS
2156,MSC TAMPICO,MSCTAMPICO,MSCTAMPICOV
9176,ZIM DANUBE,ZIMDANUBE,DANUBE
9711,ZIM SPARROW,ZIMSPARROW,SPARROW
10079,ZIM GANGES,ZIMGANGES,GANGES
19464,MSC RANIA VIII,MSCRANIAVIII,MSCRANIA


## 4. 같은 선박명끼리 후보 연결 → 입항·출항 차이 확인
이 단계의 merge 결과는 후보입니다. 같은 선박의 다른 날짜 항차도 포함되므로 다음 셀에서 시간 조건을 적용합니다.

In [4]:
candidate = r.merge(p, on='대응키', suffixes=('_record','_port1'))
candidate['입항차이_시간'] = (candidate['입항']-candidate['접안']).dt.total_seconds()/3600
candidate['출항차이_시간'] = (candidate['출항']-candidate['터미널출항']).dt.total_seconds()/3600
cols = ['rid','pid','선명_record','모선항차','입항','접안','출항','터미널출항','입항차이_시간','출항차이_시간']
display(candidate.loc[candidate['출항차이_시간'].abs().le(6), cols].head(10))

,rid,pid,선명_record,모선항차,입항,접안,출항,터미널출항,입항차이_시간,출항차이_시간
0,3,1783,MSC JEWEL,MJEW008,2022-01-01 03:22:00,2022-01-01 03:40:00,2022-01-03 05:00:00,2022-01-03 05:00:00,-0.300000,0.000000
11,7,1784,COSCO SAO PAULO,COSP001,2022-01-01 07:10:00,2022-01-01 07:20:00,2022-01-02 06:55:00,2022-01-02 07:00:00,-0.166667,-0.083333
15,17,1785,COPIAPO,COPX001,2022-01-02 08:15:00,2022-01-02 08:10:00,2022-01-04 12:05:00,2022-01-04 12:00:00,0.083333,0.083333
21,18,1786,MSC ELAINE,MELN003,2022-01-02 08:58:00,2022-01-02 09:10:00,2022-01-03 19:55:00,2022-01-03 20:00:00,-0.200000,-0.083333
24,30,1787,MSC DESIREE,MDES002,2022-01-03 07:10:00,2022-01-03 07:20:00,2022-01-04 20:20:00,2022-01-04 20:00:00,-0.166667,0.333333
27,39,1788,CROATIA,OROA001,2022-01-03 21:57:00,2022-01-03 22:20:00,2022-01-05 05:05:00,2022-01-05 05:00:00,-0.383333,0.083333
28,48,1789,XIN BIN ZHOU,MXBZ013,2022-01-04 14:00:00,2022-01-04 14:05:00,2022-01-05 12:00:00,2022-01-05 12:00:00,-0.083333,0.000000
37,57,1792,MSC NORA,MNOA001,2022-01-05 08:43:00,2022-01-05 09:00:00,2022-01-06 02:15:00,2022-01-06 02:00:00,-0.283333,0.250000
43,58,1793,MSC ELISA,MSEL009,2022-01-05 09:05:00,2022-01-05 09:20:00,2022-01-07 09:32:00,2022-01-07 09:00:00,-0.250000,0.533333
48,66,1795,CONTI ANNAPURNA,OCNO001,2022-01-06 04:15:00,2022-01-06 04:30:00,2022-01-08 07:25:00,2022-01-08 07:00:00,-0.250000,0.416667


## 5. 시간 조건으로 대응 추출 → 다중 항차·미대응 확인
겹치는 기간이 있고, 입항 또는 출항 차이가 6시간 이내이거나 터미널 구간이 record 기간 안에 포함되는 후보를 남깁니다. 포함 판정의 경계 허용은 30분입니다. 한 터미널 항차가 여러 record에 붙는 경우 검증에서 중단합니다.

In [5]:
inside = (candidate['접안'] >= candidate['입항']-pd.Timedelta(minutes=30)) & (candidate['터미널출항'] <= candidate['출항']+pd.Timedelta(minutes=30))
overlap = (candidate['터미널출항'] > candidate['입항']) & (candidate['접안'] < candidate['출항'])
near = candidate['입항차이_시간'].abs().le(6) | candidate['출항차이_시간'].abs().le(6)
links = candidate[overlap & (inside | near)].copy()
assert links.groupby('pid')['rid'].nunique().max()==1
counts = links.groupby('rid')['pid'].nunique()
display(counts.value_counts().rename_axis('대응 항차 수').to_frame('record 건수'))
unmatched = r[~r['rid'].isin(links['rid'])].copy()
display(unmatched[['선명','입항','출항','계선장소']])
print('미대응:',len(unmatched))

,record 건수
대응 항차 수,
1,2896
2,16


,선명,입항,출항,계선장소
1757,PACIFIC BUSAN,2022-05-10 03:13:00,2022-05-11 04:00:00,신항 1부두 2선석
5628,TAMINA,2023-02-17 18:00:00,2023-02-21 15:58:00,신항 1부두 1선석
6987,MSC COLETTE III,2023-05-23 05:05:00,2023-05-23 14:01:00,신항 1부두 3선석
12831,RUN LONG,2024-07-06 21:15:00,2024-07-07 15:41:00,신항 1부두 1선석
13810,PACIFIC TIANJIN,2024-09-15 15:00:00,2024-09-16 15:42:00,신항 1부두 3선석
15250,NYK FUSHIMI,2025-01-02 02:20:00,2025-01-03 01:52:00,신항 1부두 2선석
16006,MSC FLAVIA,2025-03-03 23:17:00,2025-03-06 11:58:00,신항 1부두 3선석
19103,MSC CAIRO IV,2025-10-26 20:20:00,2025-10-27 17:18:00,신항 1부두 1선석


미대응: 8


## 6. 두 항차가 연결된 16건 추출 → 32행을 눈으로 확인
완전 포함 8건과 30분 경계 허용 16건을 별도로 계산합니다. 이 숫자는 미대응 8건과 다른 개념입니다.

In [6]:
split_ids = counts[counts==2].index
split_rows = links[links['rid'].isin(split_ids)].sort_values(['rid','접안'])
strict = links[(links['접안']>=links['입항']) & (links['터미널출항']<=links['출항'])].groupby('rid')['pid'].nunique()
loose = links[(links['접안']>=links['입항']-pd.Timedelta(minutes=30)) & (links['터미널출항']<=links['출항']+pd.Timedelta(minutes=30))].groupby('rid')['pid'].nunique()
print('완전 포함:', (strict>=2).sum(), '30분 허용:', (loose>=2).sum())
print('분할 대상:', len(split_ids), '분할 후:',len(split_rows))
display(split_rows[cols])
assert len(split_ids)==16 and len(split_rows)==32 and len(unmatched)==8

,rid,pid,선명_record,모선항차,입항,접안,출항,터미널출항,입항차이_시간,출항차이_시간
1217,853,1935,XIN BIN ZHOU,MXBZ005,2022-03-05 19:48:00,2022-03-05 20:20:00,2022-03-09 23:58:00,2022-03-06 15:00:00,-0.533333,80.966667
1218,853,1943,XIN BIN ZHOU,MXBZ006,2022-03-05 19:48:00,2022-03-09 14:00:00,2022-03-09 23:58:00,2022-03-10 00:00:00,-90.200000,-0.033333
3408,2576,2210,MSC MUNDRA,MMDU001,2022-07-09 17:00:00,2022-07-09 17:05:00,2022-07-15 11:52:00,2022-07-10 17:00:00,-0.083333,114.866667
3409,2576,2217,MSC MUNDRA,MMDU002,2022-07-09 17:00:00,2022-07-13 22:05:00,2022-07-15 11:52:00,2022-07-15 12:00:00,-101.083333,-0.133333
4192,3014,2274,MSC LA SPEZIA,MLPZ001,2022-08-11 00:15:00,2022-08-11 00:30:00,2022-08-14 11:55:00,2022-08-12 00:00:00,-0.250000,59.916667
4193,3014,2280,MSC LA SPEZIA,MLPZ002,2022-08-11 00:15:00,2022-08-13 05:00:00,2022-08-14 11:55:00,2022-08-14 12:00:00,-52.750000,-0.083333
9372,5989,2724,MSC ALDI III,MADI002,2023-03-15 10:47:00,2023-03-15 11:00:00,2023-03-17 13:00:00,2023-03-16 01:00:00,-0.216667,36.000000
9373,5989,2728,MSC ALDI III,MADI003,2023-03-15 10:47:00,2023-03-16 20:40:00,2023-03-17 13:00:00,2023-03-17 13:00:00,-33.883333,0.000000
16718,8946,3248,BOTANY,MBON004,2023-10-01 23:20:00,2023-10-01 23:40:00,2023-10-07 07:54:00,2023-10-02 21:00:00,-0.333333,106.900000
16719,8946,3262,BOTANY,MBON005,2023-10-01 23:20:00,2023-10-07 00:00:00,2023-10-07 07:54:00,2023-10-07 08:00:00,-120.666667,-0.100000


완전 포함: 8 30분 허용: 16
분할 대상: 16 분할 후: 32


## 7. 원본 행 복사 후 네 열 변경 → 변경 전후 확인

In [7]:
new_rows=[]
changes=[]
for _, link in split_rows.iterrows():
    original = record.loc[link['rid']].copy()
    changed = original.copy()
    changed['입항일시'] = str(link['접안'])
    changed['출항일시'] = str(link['터미널출항'])
    berth = pd.Series([link['선석']]).str.extract(r'T(\d+)')[0].iloc[0]
    changed['계선장소'] = f'신항 1부두 {berth}선석'
    changed['재항시간'] = str((link['터미널출항']-link['접안']).total_seconds()/3600)
    other = record.columns.difference(['입항일시','출항일시','계선장소','재항시간'])
    assert original[other].equals(changed[other])
    new_rows.append(changed)
    changes.append({'선명':original['선명'],'모선항차':link['모선항차'],'기존입항':original['입항일시'],'새입항':changed['입항일시'],'기존출항':original['출항일시'],'새출항':changed['출항일시'],'새계선장소':changed['계선장소'],'새재항시간':changed['재항시간']})
new_rows = pd.DataFrame(new_rows)
change_table = pd.DataFrame(changes)
display(change_table)

,선명,모선항차,기존입항,새입항,기존출항,새출항,새계선장소,새재항시간
0,XIN BIN ZHOU,MXBZ005,2022-03-05 19:48:00,2022-03-05 20:20:00,2022-03-09 23:58:00,2022-03-06 15:00:00,신항 1부두 1선석,18.666666666666668
1,XIN BIN ZHOU,MXBZ006,2022-03-05 19:48:00,2022-03-09 14:00:00,2022-03-09 23:58:00,2022-03-10 00:00:00,신항 1부두 1선석,10.0
2,MSC MUNDRA,MMDU001,2022-07-09 17:00:00,2022-07-09 17:05:00,2022-07-15 11:52:00,2022-07-10 17:00:00,신항 1부두 1선석,23.916666666666668
3,MSC MUNDRA,MMDU002,2022-07-09 17:00:00,2022-07-13 22:05:00,2022-07-15 11:52:00,2022-07-15 12:00:00,신항 1부두 2선석,37.916666666666664
4,MSC LA SPEZIA,MLPZ001,2022-08-11 00:15:00,2022-08-11 00:30:00,2022-08-14 11:55:00,2022-08-12 00:00:00,신항 1부두 3선석,23.5
5,MSC LA SPEZIA,MLPZ002,2022-08-11 00:15:00,2022-08-13 05:00:00,2022-08-14 11:55:00,2022-08-14 12:00:00,신항 1부두 3선석,31.0
6,MSC ALDI III,MADI002,2023-03-15 10:47:00,2023-03-15 11:00:00,2023-03-17 13:00:00,2023-03-16 01:00:00,신항 1부두 2선석,14.0
7,MSC ALDI III,MADI003,2023-03-15 10:47:00,2023-03-16 20:40:00,2023-03-17 13:00:00,2023-03-17 13:00:00,신항 1부두 2선석,16.333333333333332
8,BOTANY,MBON004,2023-10-01 23:20:00,2023-10-01 23:40:00,2023-10-07 07:54:00,2023-10-02 21:00:00,신항 1부두 2선석,21.333333333333332
9,BOTANY,MBON005,2023-10-01 23:20:00,2023-10-07 00:00:00,2023-10-07 07:54:00,2023-10-07 08:00:00,신항 1부두 3선석,8.0


## 8. 기존 16행·미대응 8행 제외 후 32행 추가 → 최종 확인
분할하지 않은 record 행은 원래 값을 유지합니다. 나머지 부두도 유지합니다.

In [ ]:
drop_ids = set(split_ids) | set(unmatched['rid'])
result = pd.concat([record.drop(index=list(drop_ids)),new_rows],ignore_index=True)
result = result.iloc[pd.to_datetime(result['입항일시']).argsort()].reset_index(drop=True)
result_pier1 = result[result['계선장소'].str.contains('신항 1부두')]
print('전체:',len(result),'1부두:',len(result_pier1))
display(result_pier1.head(5))
assert len(result)==19989 and len(result_pier1)==2928

## 9. port1에만 남은 항차 제외 → 양쪽 개수 확인

In [9]:
port_excluded = p[~p['pid'].isin(links['pid'])]
display(port_excluded[['선명','모선항차','접안','터미널출항']].head(20))
port_result = port1.loc[sorted(links['pid'].unique())].copy()
print('port1에서만 존재:',len(port_excluded))
display(pd.DataFrame({'자료':['record 1부두','port1'],'행 수':[len(result_pier1),len(port_result)]}))
assert len(port_excluded)==271 and len(port_result)==2928
assert links['pid'].nunique()==len(links)
assert port_result.equals(port1.loc[sorted(links['pid'].unique())])

,선명,모선항차,접안,터미널출항
1790,MSC KYMEA II,MKYM001,2022-01-04 14:05:00,2022-01-05 06:00:00
1791,MSC RICCARDA II,MRIA001,2022-01-04 22:45:00,2022-01-05 07:00:00
1794,COSCO FOS,CFOS018,2022-01-05 14:00:00,2022-01-07 09:00:00
1808,MSC TIPHAINE,MTPH001,2022-01-11 06:00:00,2022-01-11 18:00:00
1816,MSC IDA II,MIDA001,2022-01-13 14:00:00,2022-01-14 02:00:00
1820,COSCO FOS,CFOS001,2022-01-16 02:20:00,2022-01-17 09:00:00
1828,MSC TIPHAINE,MTPH002,2022-01-18 13:03:00,2022-01-19 01:00:00
1840,HEUNG-A ULSAN,HUSN004,2022-01-22 19:10:00,2022-01-23 01:00:00
1842,COSCO FOS,CFOS002,2022-01-23 04:00:00,2022-01-24 06:00:00
1863,COSCO FOS,CFOS003,2022-02-01 10:00:00,2022-02-03 11:00:00


,자료,행 수
0,record 1부두,2928
1,port1,2928


port1에서만 존재: 271


## 10. 확인한 결과 저장
아래 셀을 실행하면 수정본 두 개와 변경·제외 목록이 저장됩니다.

In [10]:
# result.to_csv(OUT/'record.csv',index=False,encoding='utf-8-sig')
# port_result.to_csv(OUT/'port1.csv',index=False,encoding='utf-8-sig')
# change_table.to_csv(OUT/'분할_변경내역.csv',index=False,encoding='utf-8-sig')
# unmatched[record.columns].to_csv(OUT/'record_제외8건.csv',index=False,encoding='utf-8-sig')
# port_excluded[port1.columns].to_csv(OUT/'port1_제외271건.csv',index=False,encoding='utf-8-sig')
# print('저장 위치:',OUT.resolve())

저장 위치: /workspace/scratch/e27fd41c5f22/tmp/step_verify
